# 01 — Getting Started with OpenCV

Load (or generate) an image, display it, and try a few basic operations.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

IMAGES = Path("..") / "images"
print("OpenCV", cv2.__version__)

In [ ]:
def show(*imgs, titles=None, cmap=None):
    """Display one or more images side by side. Converts BGR to RGB for colour images."""
    fig, axes = plt.subplots(1, len(imgs), figsize=(5 * len(imgs), 5))
    axes = np.atleast_1d(axes)
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        if img.ndim == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap=cmap or "gray")
        ax.set_title(titles[i] if titles else "")
        ax.axis("off")
    plt.show()

## All images

Every image in `images/`, shown as a grid.

In [ ]:
import math
import re

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def natural_key(p):
    """Sort 'Face 2' before 'Face 10'."""
    return [int(s) if s.isdigit() else s.lower() for s in re.split(r"(\d+)", p.stem)]

def show_grid(paths, cols=4, size=3):
    """Display images from a list of paths in a grid with cols columns."""
    rows = math.ceil(len(paths) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, p in zip(axes.flat, paths):
        im = cv2.imread(str(p))
        if im is None:
            ax.set_title(f"{p.name}
(unreadable)")
            continue
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(p.name)
    plt.tight_layout()
    plt.show()

image_paths = sorted((p for p in IMAGES.iterdir() if p.suffix.lower() in EXTS), key=natural_key)
print(f"{len(image_paths)} images found")
show_grid(image_paths)

## Load an image

Drop any image into `images/` and set `name` below. If none is found, a synthetic test image is drawn instead.

In [ ]:
name = "sample.jpg"
img = cv2.imread(str(IMAGES / name))

if img is None:
    img = np.full((400, 600, 3), 30, np.uint8)
    cv2.rectangle(img, (50, 50), (250, 250), (0, 200, 255), -1)
    cv2.circle(img, (420, 160), 100, (255, 120, 0), -1)
    cv2.line(img, (50, 330), (550, 330), (0, 255, 0), 8)
    cv2.putText(img, "OpenCV", (200, 390), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)

print(img.shape, img.dtype)
show(img, titles=["original"])

## Basic operations

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (7, 7), 0)
edges = cv2.Canny(blur, 50, 150)

show(gray, blur, edges, titles=["grayscale", "blurred", "canny edges"])

In [ ]:
_, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

annotated = img.copy()
cv2.drawContours(annotated, contours, -1, (0, 0, 255), 3)
print(f"{len(contours)} contours found")
show(thresh, annotated, titles=["otsu threshold", "contours"])